# HW02 Part 2: Build an S3 Data Lake

This notebook creates a private S3 bucket and uploads the hourly Wikipedia Parquet files from EC2 into the `wikipedia-hourly` prefix.

In [1]:
import boto3
from pathlib import Path

## Set up the S3 bucket

Define the bucket name and create an S3 client for interacting with AWS.

In [3]:
bucket_name = "dsan6000-jt1573"
prefix = "wikipedia-hourly/"

s3 = boto3.client("s3")

## Create the S3 bucket

Create a private S3 bucket for storing the Wikipedia hourly event data.

In [4]:
s3.create_bucket(Bucket=bucket_name)

print(f"Created bucket: s3://{bucket_name}")

Created bucket: s3://dsan6000-jt1573


## Verify the bucket

Check that the new S3 bucket exists in the current AWS account.

In [5]:
response = s3.list_buckets()

bucket_names = [
    bucket["Name"]
    for bucket in response["Buckets"]
]

bucket_name in bucket_names

True

## Find the local Parquet files

Locate the hourly Parquet files previously downloaded to the local `data` directory.

In [6]:
data_dir = Path("data")

parquet_files = sorted(data_dir.glob("*.parquet"))

print(f"Found {len(parquet_files)} local Parquet files.")
parquet_files[:5]

Found 24 local Parquet files.


[PosixPath('data/20260901_040000.parquet'),
 PosixPath('data/20260901_050000.parquet'),
 PosixPath('data/20260901_060000.parquet'),
 PosixPath('data/20260901_070000.parquet'),
 PosixPath('data/20260901_080000.parquet')]

## Upload the Parquet files to the data lake

Upload each local Parquet file to the `wikipedia-hourly` prefix in the new S3 bucket.

In [7]:
for local_file in parquet_files:
    s3_key = prefix + local_file.name

    s3.upload_file(
        str(local_file),
        bucket_name,
        s3_key
    )

print(f"Uploaded {len(parquet_files)} Parquet files to s3://{bucket_name}/{prefix}")

Uploaded 24 Parquet files to s3://dsan6000-jt1573/wikipedia-hourly/


## Verify the uploaded files

List the objects stored under the `wikipedia-hourly` prefix and confirm that all local Parquet files were uploaded.

In [8]:
response = s3.list_objects_v2(
    Bucket=bucket_name,
    Prefix=prefix
)

uploaded_files = [
    obj["Key"]
    for obj in response.get("Contents", [])
    if obj["Key"].endswith(".parquet")
]

print(f"Number of uploaded files: {len(uploaded_files)}")
uploaded_files[:5]

assert len(uploaded_files) == len(parquet_files)

Number of uploaded files: 24
